# ImageFolder Biological Split

Split `DataSet_Training` by biological sample, keeping every timestamp and image from a sample together. For example, `VID1730_E2_1_00d06h00m` and `VID1730_E2_1_2023y05m17d_20h45m` both belong to **VID1730_E2_1**.

Configure the number of folds and choose a validation fold below. Each sample is assigned to exactly one validation fold; the other folds form training data. Samples are stratified by class, so each fold has approximately the same number of biological samples per class. Image counts can differ because samples contain different numbers of crops.

The input layout is `DataSet_Training/<class>/<experiment>/<sample>_<timestamp>/<images>`. Class names are normalized to lowercase for compatibility with the ConvNeXt training notebooks. The original data stays in place.


In [11]:
from pathlib import Path
from hashlib import sha256
import json
import random
import re
import shutil

import pandas as pd
from IPython.display import display
from torchvision.datasets import ImageFolder
from torchvision.datasets.folder import IMG_EXTENSIONS


def find_repo_root():
    candidate = Path.cwd().resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Run this notebook inside the DeathScope repository.")


repo_root = find_repo_root()


## Configuration

`N_FOLDS=5` gives approximately 80% of biological samples for training and 20% for validation. Set `VALIDATION_FOLD` to any integer from 1 to `N_FOLDS`. Changing only this setting selects another held-out fold while keeping sample assignments fixed.

By default only the selected fold is exported. Set `EXPORT_ALL_FOLDS=True` to export every fold for cross-validation. `COPY_FILES=True` produces portable copies; `False` creates symlinks that require the original source paths to remain available. Use a new `OUTPUT_ROOT` when changing fold assignments; existing incompatible outputs are rejected to prevent mixing splits.


In [7]:
SOURCE_ROOT = repo_root / "training/ConvNeXt/DataSet_Training"
OUTPUT_ROOT = repo_root / "training/ConvNeXt/ImageFolder_Biolgical_split"
N_FOLDS = 5
VALIDATION_FOLD = 1  # 1 .. N_FOLDS
SEED = 42
EXPORT_ALL_FOLDS = True
COPY_FILES = True

# First three components are the biological sample; everything after is timestamp.
# Supports elapsed and calendar timestamps, e.g. VID1730_E2_1_<timestamp>.
SAMPLE_FOLDER_PATTERN = r"^(?P<sample_id>VID\d+_[^_]+_\d+)_(?P<timestamp>.+)$"

if not isinstance(N_FOLDS, int) or isinstance(N_FOLDS, bool) or N_FOLDS < 2:
    raise ValueError("N_FOLDS must be an integer >= 2.")
if not isinstance(VALIDATION_FOLD, int) or isinstance(VALIDATION_FOLD, bool) or not 1 <= VALIDATION_FOLD <= N_FOLDS:
    raise ValueError("VALIDATION_FOLD must be between 1 and N_FOLDS.")
SOURCE_ROOT = Path(SOURCE_ROOT).resolve()
OUTPUT_ROOT = Path(OUTPUT_ROOT).resolve()
if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_ROOT}")
if OUTPUT_ROOT == SOURCE_ROOT or SOURCE_ROOT in OUTPUT_ROOT.parents:
    raise ValueError("OUTPUT_ROOT must be outside SOURCE_ROOT.")
print(f"Source: {SOURCE_ROOT}")
print(f"Output: {OUTPUT_ROOT}")
print(f"Validation fold: {VALIDATION_FOLD} / {N_FOLDS}")


Source: /media/capp/TMP2/DeathScope/training/ConvNeXt/DataSet_Training
Output: /media/capp/TMP2/DeathScope/training/ConvNeXt/ImageFolder_Biolgical_split
Validation fold: 1 / 5


## Discover Biological Samples

The sample ID is global: experiment folders and timestamps cannot cause one sample to be split across training and validation. Unrecognized image paths or a sample ID appearing in multiple classes produce an error instead of a potentially leaking split.


In [8]:
def discover_images(source_root, folder_pattern):
    pattern = re.compile(folder_pattern)
    extensions = set(IMG_EXTENSIONS)
    records = []
    invalid = []
    class_dirs = sorted(path for path in source_root.iterdir() if path.is_dir())
    class_names = [path.name.lower() for path in class_dirs]
    if len(class_names) != 2 or len(set(class_names)) != 2:
        raise ValueError(f"Expected exactly two distinct class folders; found {class_names}.")
    for class_dir in class_dirs:
        for image_path in sorted(class_dir.rglob("*")):
            if not image_path.is_file() or image_path.suffix.lower() not in extensions:
                continue
            relative = image_path.relative_to(source_root)
            sample_folders = [(part, pattern.fullmatch(part)) for part in relative.parts[1:-1]]
            matches = [(part, match) for part, match in sample_folders if match is not None]
            if len(matches) != 1:
                invalid.append(str(relative))
                continue
            folder_name, match = matches[0]
            records.append({
                "class_name": class_dir.name.lower(),
                "sample_id": match.group("sample_id"),
                "timestamp": match.group("timestamp"),
                "timestamp_folder": folder_name,
                "source_relative_path": relative.as_posix(),
                "source_path": str(image_path),
            })
    if invalid:
        raise ValueError(
            f"{len(invalid)} image paths do not contain exactly one sample_timestamp folder. "
            f"Examples: {invalid[:5]}. Adjust SAMPLE_FOLDER_PATTERN if needed."
        )
    if not records:
        raise ValueError(f"No supported images found under {source_root}.")
    images = pd.DataFrame(records).sort_values(["class_name", "sample_id", "source_relative_path"]).reset_index(drop=True)
    conflicting = images.groupby("sample_id")["class_name"].nunique()
    conflicting = conflicting[conflicting > 1]
    if len(conflicting):
        raise ValueError(
            f"Sample IDs appear in multiple classes: {conflicting.index.tolist()}. "
            "Resolve ambiguous biological identities before stratifying."
        )
    missing_classes = set(class_names) - set(images["class_name"])
    if missing_classes:
        raise ValueError(f"Class folders contain no supported images: {sorted(missing_classes)}")
    return images


images_df = discover_images(SOURCE_ROOT, SAMPLE_FOLDER_PATTERN)
samples_df = images_df.groupby(["class_name", "sample_id"], as_index=False).agg(
    images=("source_path", "size"),
    timestamps=("timestamp", "nunique"),
)
display(samples_df.groupby("class_name").agg(biological_samples=("sample_id", "size"), images=("images", "sum")))
display(samples_df.head(15))
print(f"Found {len(images_df):,} images from {len(samples_df)} biological samples.")


,biological_samples,images
class_name,,
apoptosis,30,44105
pyroptosis,36,56385


,class_name,sample_id,images,timestamps
0,apoptosis,VID1651_F4_1,743,11
1,apoptosis,VID1651_F4_2,804,11
2,apoptosis,VID1651_F5_1,517,11
3,apoptosis,VID1651_F5_2,729,11
4,apoptosis,VID1651_F6_1,496,11
5,apoptosis,VID1651_F6_2,671,11
6,apoptosis,VID1730_E1_1,557,11
7,apoptosis,VID1730_E1_2,518,11
8,apoptosis,VID1730_E2_1,614,11
9,apoptosis,VID1730_E2_2,356,11


Found 100,490 images from 66 biological samples.


## Assign Folds And Check Separation

Assignment is deterministic for a fixed source list, seed, and number of folds. For each class, shuffled biological samples are distributed across folds; no individual image is split independently. Every fold is checked for disjoint biological sample IDs and complete image coverage.


In [9]:
def assign_sample_folds(samples, n_folds, seed):
    result = samples.copy()
    rng = random.Random(seed)
    assignments = {}
    for class_name, class_samples in result.groupby("class_name", sort=True):
        sample_ids = sorted(class_samples["sample_id"].tolist())
        if len(sample_ids) < n_folds:
            raise ValueError(
                f"Class {class_name!r} has {len(sample_ids)} biological samples; "
                f"at least {n_folds} are required. Reduce N_FOLDS."
            )
        rng.shuffle(sample_ids)
        for index, sample_id in enumerate(sample_ids):
            assignments[sample_id] = index % n_folds + 1
    result["validation_fold"] = result["sample_id"].map(assignments).astype(int)
    return result


def select_fold(images, fold_number):
    selected = images.copy()
    selected["split"] = selected["validation_fold"].map(
        lambda assigned: "val" if assigned == fold_number else "train"
    )
    train_ids = set(selected.loc[selected["split"] == "train", "sample_id"])
    val_ids = set(selected.loc[selected["split"] == "val", "sample_id"])
    if train_ids & val_ids:
        raise RuntimeError("Biological sample leakage detected.")
    if len(selected) != len(images) or selected["source_path"].duplicated().any():
        raise RuntimeError("Image coverage or uniqueness check failed.")
    for split_name in ("train", "val"):
        if set(selected.loc[selected["split"] == split_name, "class_name"]) != set(images["class_name"]):
            raise RuntimeError(f"Fold {fold_number}/{split_name} is missing a class.")
    return selected


samples_df = assign_sample_folds(samples_df, N_FOLDS, SEED)
images_df = images_df.merge(samples_df[["sample_id", "validation_fold"]], on="sample_id", validate="many_to_one")
fold_summaries = []
for fold_number in range(1, N_FOLDS + 1):
    selected = select_fold(images_df, fold_number)
    summary = selected.groupby(["split", "class_name"], as_index=False).agg(
        biological_samples=("sample_id", "nunique"), images=("source_path", "size")
    )
    summary.insert(0, "fold", fold_number)
    fold_summaries.append(summary)
fold_summary_df = pd.concat(fold_summaries, ignore_index=True)
display(fold_summary_df)
selected_df = select_fold(images_df, VALIDATION_FOLD)
display(samples_df.sort_values(["validation_fold", "class_name", "sample_id"]))
print("Verified: no biological sample occurs in both train and val in any fold.")


,fold,split,class_name,biological_samples,images
0,1,train,apoptosis,24,36918
1,1,train,pyroptosis,28,45045
2,1,val,apoptosis,6,7187
3,1,val,pyroptosis,8,11340
4,2,train,apoptosis,24,33232
5,2,train,pyroptosis,29,46582
6,2,val,apoptosis,6,10873
7,2,val,pyroptosis,7,9803
8,3,train,apoptosis,24,35482
9,3,train,pyroptosis,29,45933


,class_name,sample_id,images,timestamps,validation_fold
6,apoptosis,VID1730_E1_1,557,11,1
8,apoptosis,VID1730_E2_1,614,11,1
9,apoptosis,VID1730_E2_2,356,11,1
17,apoptosis,VID2232_G3_2,1780,11,1
19,apoptosis,VID2335_B1_2,739,10,1
...,...,...,...,...,...
34,pyroptosis,VID2232_D3_1,2285,11,5
39,pyroptosis,VID2232_D8_2,2238,11,5
43,pyroptosis,VID2335_D4_2,1280,10,5
46,pyroptosis,VID2335_E4_1,1258,10,5


Verified: no biological sample occurs in both train and val in any fold.


## Export ImageFolder

Outputs use `OUTPUT_ROOT/fold_1/train/<class>/` and `OUTPUT_ROOT/fold_1/val/<class>/` (and other chosen folds). Filenames include a hash of the original relative path to avoid crop-name collisions. Each fold saves an image manifest, biological sample manifest, summary, and configuration.

Rerunning the same plan resumes missing files. An existing output with a different source, fold assignment, or copy mode is rejected; choose a new output directory in that case. This cell may copy many images when exporting all folds.


In [10]:
def destination_name(relative_path):
    relative = Path(relative_path)
    digest = sha256(relative_path.encode("utf-8")).hexdigest()[:16]
    return f"{relative.stem}_{digest}{relative.suffix.lower()}"


def export_fold(images, samples, fold_number, output_root, source_root, n_folds, seed, copy_files):
    selected = select_fold(images, fold_number)
    fold_root = output_root / f"fold_{fold_number}"
    selected["output_relative_path"] = selected.apply(
        lambda row: f"{row['split']}/{row['class_name']}/{destination_name(row['source_relative_path'])}", axis=1
    )
    if selected["output_relative_path"].duplicated().any():
        raise RuntimeError("Destination filename collision detected.")
    config = {
        "source_root": str(source_root), "n_folds": n_folds,
        "validation_fold": fold_number, "seed": seed, "copy_files": copy_files,
        "sample_folder_pattern": SAMPLE_FOLDER_PATTERN,
    }
    plan = {"config": config, "images": selected[["source_relative_path", "sample_id", "split", "output_relative_path"]].to_dict("records")}
    config["plan_sha256"] = sha256(json.dumps(plan, sort_keys=True).encode("utf-8")).hexdigest()
    marker = fold_root / "split_config.json"
    if fold_root.exists() and any(fold_root.iterdir()):
        if not marker.is_file() or json.loads(marker.read_text()) != config:
            raise FileExistsError(f"{fold_root} contains a different or unrecognized split. Choose a new OUTPUT_ROOT.")
        expected = set(selected["output_relative_path"])
        actual = {path.relative_to(fold_root).as_posix()
                  for split_name in ("train", "val")
                  for path in (fold_root / split_name).rglob("*")
                  if path.is_file() or path.is_symlink()}
        if actual - expected:
            raise FileExistsError(f"Unexpected files in {fold_root}; choose a new OUTPUT_ROOT.")
    fold_root.mkdir(parents=True, exist_ok=True)
    marker.write_text(json.dumps(config, indent=2) + "\n")
    for row in selected.itertuples(index=False):
        src = Path(row.source_path)
        dst = fold_root / row.output_relative_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            if copy_files and dst.stat().st_size != src.stat().st_size:
                raise ValueError(f"Existing copy has a different size: {dst}")
            if not copy_files and (not dst.is_symlink() or dst.resolve() != src.resolve()):
                raise ValueError(f"Existing symlink points to a different source: {dst}")
            continue
        if dst.is_symlink():
            raise ValueError(f"Broken symlink at {dst}; use a fresh output directory.")
        if copy_files:
            shutil.copy2(src, dst)
        else:
            dst.symlink_to(src.resolve())
    # Verify the exported dataset is immediately readable by training notebooks.
    for split_name in ("train", "val"):
        dataset = ImageFolder(root=str(fold_root / split_name))
        expected_count = int((selected["split"] == split_name).sum())
        if len(dataset) != expected_count:
            raise RuntimeError(f"Export count mismatch in {fold_root}/{split_name}.")
    selected.to_csv(fold_root / "image_manifest.csv", index=False)
    fold_samples = samples.copy()
    fold_samples["split"] = fold_samples["validation_fold"].map(
        lambda assigned: "val" if assigned == fold_number else "train"
    )
    fold_samples.to_csv(fold_root / "biological_sample_manifest.csv", index=False)
    print(f"Fold {fold_number}: exported {len(selected):,} images to {fold_root}")
    return fold_root


folds_to_export = list(range(1, N_FOLDS + 1)) if EXPORT_ALL_FOLDS else [VALIDATION_FOLD]
exported_roots = [
    export_fold(images_df, samples_df, number, OUTPUT_ROOT, SOURCE_ROOT, N_FOLDS, SEED, COPY_FILES)
    for number in folds_to_export
]
samples_df.to_csv(OUTPUT_ROOT / "sample_fold_assignments.csv", index=False)
fold_summary_df.to_csv(OUTPUT_ROOT / "fold_summary.csv", index=False)
SELECTED_IMAGEFOLDER = OUTPUT_ROOT / f"fold_{VALIDATION_FOLD}"
print(f"Selected ImageFolder for training: {SELECTED_IMAGEFOLDER}")


Fold 1: exported 100,490 images to /media/capp/TMP2/DeathScope/training/ConvNeXt/ImageFolder_Biolgical_split/fold_1
Fold 2: exported 100,490 images to /media/capp/TMP2/DeathScope/training/ConvNeXt/ImageFolder_Biolgical_split/fold_2
Fold 3: exported 100,490 images to /media/capp/TMP2/DeathScope/training/ConvNeXt/ImageFolder_Biolgical_split/fold_3
Fold 4: exported 100,490 images to /media/capp/TMP2/DeathScope/training/ConvNeXt/ImageFolder_Biolgical_split/fold_4
Fold 5: exported 100,490 images to /media/capp/TMP2/DeathScope/training/ConvNeXt/ImageFolder_Biolgical_split/fold_5
Selected ImageFolder for training: /media/capp/TMP2/DeathScope/training/ConvNeXt/ImageFolder_Biolgical_split/fold_1


## Train With This Split

In `ConvNeXt-Large_training.ipynb`, set:

```python
SPLIT_ROOT = repo_root / "training/ConvNeXt/ImageFolder_Biolgical_split/fold_1"
USE_EXISTING_SPLIT = True
```

Replace `fold_1` with the chosen `VALIDATION_FOLD`. Restart the kernel and run the training notebook from the beginning.

For all-fold training, first export with `EXPORT_ALL_FOLDS=True`, then in `ConvNeXt-Large_training_5fold_cross_validation.ipynb` set:

```python
SPLIT_ROOT = repo_root / "training/ConvNeXt/ImageFolder_Biolgical_split"
K_FOLDS = 5  # Must equal N_FOLDS used here.
REUSE_EXISTING_FOLDS = True
```

The five-fold training notebook will use these biological fold directories directly. Do not pool and re-split them by individual image.
